# Diffusion-Based AoA/Amplitude/Phase Reconstruction — Full GPU Training (54k samples, 100 epochs)

This notebook trains a **12-channel DDPM** on ray-traced radio maps and runs
**DPS (Diffusion Posterior Sampling)** inference for channel map reconstruction.

| Setting | Value |
|---|---|
| Channels | 12: AoA(3) + Amp(3) + sin(phase)(3) + cos(phase)(3) |
| Phase | Coarse-grained, N=64, lambda_eff=8m, ~16 cycles/map |
| Buildings | 60 configs (20×1-bldg + 20×2-bldg + 20×3-bldg) |
| BS positions | **30 × 30 = 900 per layout** (bs_grid_spacing = 4 m) |
| Total samples | **54,000** |
| Resolution | 128×128 (1 m grid) |
| Model | U-Net ~21M params, 100 diffusion steps |
| Epochs | **100** |
| **Expected training time** | **~62 h on A100** over 3 Pro+ sessions, ~186 h on T4 |
| **Drive footprint** | **~20 GB** (raw HDF5 + checkpoints) |

**Required runtime:** `Runtime → Change runtime type → A100 GPU` with **High-RAM ON**.
At 54k samples the tensor list peaks around 45 GB of system RAM, so high-RAM is **not optional**. T4 will work but takes ~186 hours over ~16 sessions.

The data generation uses a **streaming pipeline** (writes each building config to HDF5 immediately, frees RAM, continues) — peak RAM during generation stays around 1 GB regardless of dataset size. This is what makes 54,000 samples possible without OOM.

## Step 1: Check GPU and Setup Environment

In [ ]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Mount Google Drive and Extract Project

Upload `diffusion-posterior-sampling.zip` to your Google Drive root (`My Drive/`), then run the cells below.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

# === EDIT THIS PATH ===
# Change this to match where you uploaded the zip file in Google Drive
ZIP_PATH = '/content/drive/MyDrive/diffusion-posterior-sampling.zip'
# ======================

PROJECT_DIR = '/content/diffusion-posterior-sampling'

if os.path.exists(ZIP_PATH):
    print(f"Extracting {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    
    # Find the extracted folder (might be nested)
    if not os.path.exists(PROJECT_DIR):
        # Check if it extracted with a different name
        extracted = [d for d in os.listdir('/content/') if 'diffusion' in d.lower() and os.path.isdir(f'/content/{d}')]
        if extracted:
            os.rename(f'/content/{extracted[0]}', PROJECT_DIR)
            print(f"Renamed {extracted[0]} -> diffusion-posterior-sampling")
    
    print(f"Project extracted to {PROJECT_DIR}")
else:
    print(f"ZIP not found at {ZIP_PATH}")
    print("Please update ZIP_PATH to match your Google Drive location.")
    print("Example: /content/drive/MyDrive/thesis/diffusion-posterior-sampling.zip")

In [ ]:
# Change to project directory
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"\nProject files:")
for f in sorted(os.listdir('.')):
    print(f"  {f}")

In [ ]:
# Install dependencies
!pip install h5py tqdm tensorboard pyyaml matplotlib scipy PyMuPDF -q
print("All dependencies installed.")

## Step 2b: Create motionblur Stub

The diffusion library imports `motionblur` (from the original DPS image code), but we don't need it for radio map training. Create a minimal stub so the import doesn't fail.

In [ ]:
import os

# Create stub motionblur package (not needed for radio maps, but imported by DPS image utils)
os.makedirs('motionblur', exist_ok=True)
with open('motionblur/__init__.py', 'w') as f:
    f.write('# Stub - motionblur not needed for radio map training\n')
with open('motionblur/motionblur.py', 'w') as f:
    f.write('class Kernel:\n    """Stub class - not used for radio map training.\"\"\"\n    pass\n')
print("motionblur stub created successfully.")

## Step 3: Clean Old Caches

Delete any stale data from the CPU test run so the GPU generates fresh full-scale data.

In [ ]:
import glob

# Remove old cached data (from CPU test with 6 configs)
cache_patterns = [
    'data/building_training_randomized/*.h5',
    'data/building_training_randomized/*.pkl',
    'data/building_training_randomized/*.pt',
]

deleted = 0
for pattern in cache_patterns:
    for f in glob.glob(pattern):
        os.remove(f)
        print(f"Deleted: {f}")
        deleted += 1

# Remove old checkpoints from CPU test
for f in glob.glob('checkpoints/aoa_amp_building/*.pt'):
    os.remove(f)
    print(f"Deleted: {f}")
    deleted += 1

print(f"\nCleaned {deleted} stale files." if deleted else "\nNo stale files found. Clean start.")

## Step 4: Verify Config (Full-Scale)

Confirm the config has full-scale values, not the reduced CPU test values.

In [ ]:
import yaml

with open('configs/aoa_amp_building_config.yaml') as f:
    config = yaml.safe_load(f)

print("=== Key Config Values ===")
print(f"  data_channels:        {config['data_channels']}")
print(f"  batch_size:           {config['batch_size']}")
print(f"  num_epochs:           {config['num_epochs']}")
print(f"  attention_resolutions:{config['attention_resolutions']}")
print(f"  image_size:           {config['image_size']}")
ds = config['dataset']
print(f"  bs_grid_spacing:      {ds['bs_grid_spacing']}")
print(f"  num_building_sets:    {ds['num_building_sets']}")
print(f"  building_distribution:{ds['building_distribution']}")

# Sanity checks for the 54,000-sample x 100-epoch full run
assert config['data_channels'] == 12, "Should be 12 channels!"
assert config['num_epochs'] == 100, "Should be 100 epochs for this run!"
assert ds['num_building_sets'] == 60, "Should be 60 building configs!"
assert ds['bs_grid_spacing'] == 4.0, "Should be 4.0 (60 layouts x 900 BS = 54,000 samples)"

print("\nAll checks passed.")
print("  Total samples:   54,000  (60 layouts x 900 BS positions)")
print("  Training time:   ~62 h on A100 over 3 sessions  /  ~186 h on T4")
print("  RAM peak:        ~45 GB (need high-RAM A100 runtime)")
print("  Drive footprint: ~20 GB (raw HDF5 + checkpoints)")

## Step 4b: Verify Phase Representation (Quick Sanity Check)

Generate one sample map and verify the coarse-grained phase shows smooth concentric rings (not dense rainbow noise).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aoa_amp_building import RayTracingAoAMap, _PHASE_COARSEN, _WAVELENGTH

# Quick single-map test
rt = RayTracingAoAMap(map_size=(128, 128), grid_spacing=1.0)
rt.add_building(40, 60, 25, 15)
rt.add_building(70, 90, 20, 20)
rt.set_base_station(95, 80)
phase_maps = rt.generate_phase_map(num_paths=3)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, pm, t in zip(axes, phase_maps,
                     ['Strongest', '2nd Strongest', '3rd Strongest']):
    im = ax.imshow(pm.T, origin='lower', extent=[0,128,0,128],
                   cmap='hsv', vmin=-np.pi, vmax=np.pi)
    ax.set_title(f'{t} Path - Phase')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    plt.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout(); plt.show()

leff = _PHASE_COARSEN * _WAVELENGTH
n_unique = len(np.unique(np.round(phase_maps[0], 2)))
print(f"Phase coarsening: N={_PHASE_COARSEN}, lambda_eff={leff:.0f}m, ~{128/leff:.0f} cycles/map")
print(f"Strongest path unique values: {n_unique} (should be >> 1)")
if n_unique > 50:
    print("PASS: Phase maps show spatial variation (smooth concentric rings).")
else:
    print("WARNING: Phase maps may be too flat. Check _PHASE_COARSEN in aoa_amp_building.py")

## Step 4c: Auto-Save Checkpoints to Drive (IMPORTANT)

This symlinks the local `checkpoints/aoa_amp_building/` folder to a Drive folder, so **every checkpoint the training writes goes directly to your Google Drive**. No manual backups needed — if Colab disconnects, all your progress is already on Drive.

Also symlinks the data cache so the generated `.h5` file is saved to Drive too.

**Run this cell once before Step 5.** Re-running is safe (idempotent).

In [ ]:
import os, shutil

DRIVE_BASE = '/content/drive/MyDrive/thesis_results'

# --- Symlink: checkpoints/aoa_amp_building -> Drive folder ---
drive_ckpt = os.path.join(DRIVE_BASE, 'checkpoints')
local_ckpt = 'checkpoints/aoa_amp_building'

os.makedirs(drive_ckpt, exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Remove local checkpoint dir if it exists (and isn't already a symlink)
if os.path.islink(local_ckpt):
    print(f"[OK] Symlink already exists: {local_ckpt} -> {os.readlink(local_ckpt)}")
else:
    if os.path.exists(local_ckpt):
        # If there are existing checkpoints, move them to Drive first
        for f in os.listdir(local_ckpt):
            src = os.path.join(local_ckpt, f)
            dst = os.path.join(drive_ckpt, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
        shutil.rmtree(local_ckpt)
    os.symlink(drive_ckpt, local_ckpt)
    print(f"[OK] Created symlink: {local_ckpt} -> {drive_ckpt}")

# --- Symlink: data cache file -> Drive ---
drive_data = os.path.join(DRIVE_BASE, 'all_building_configs_data_42.h5')
local_data = 'data/building_training_randomized/all_building_configs_data_42.h5'

os.makedirs('data/building_training_randomized', exist_ok=True)

if os.path.islink(local_data):
    print(f"[OK] Data symlink already exists: {local_data} -> {os.readlink(local_data)}")
else:
    # If local data exists and Drive copy doesn't, copy it to Drive first
    if os.path.exists(local_data) and not os.path.exists(drive_data):
        print(f"Moving local data cache to Drive ({os.path.getsize(local_data)/1024**3:.2f} GB)...")
        shutil.copy2(local_data, drive_data)
    # Remove local file if it exists, then symlink
    if os.path.exists(local_data) and not os.path.islink(local_data):
        os.remove(local_data)
    os.symlink(drive_data, local_data)
    print(f"[OK] Created symlink: {local_data} -> {drive_data}")

# --- Verify ---
print(f"\nDrive folder: {DRIVE_BASE}")
if os.path.exists(DRIVE_BASE):
    items = os.listdir(DRIVE_BASE)
    print(f"Contents: {items if items else '(empty - will be filled during training)'}")

print("\nAll checkpoints and data will now save directly to Drive.")
print("You can skip Step 5a during training - everything is auto-backed-up.")

## Step 5: Generate Data + Train (54,000 samples × 100 epochs)

This is the main step. It will:
1. **Stream-generate 54,000 ray-traced samples** — writes each building config to HDF5 immediately, frees RAM (~1 GB peak during generation). ~**45 min** on A100.
2. **Stream tensor conversion** from raw HDF5 (no need to load self.data into RAM). ~**3 min**.
3. **Incremental tensor cache save** (one sample at a time, no RAM spike). ~**1 min**.
4. **Train** the 12-channel diffusion model for **100 epochs** — ~**62 h on A100**, ~**186 h on T4**.

**Total estimated time: ~63 h on A100 high-RAM** — typically **3 Pro+ sessions** of ~21 h each.

Checkpoints save every **5 epochs** directly to Drive via the Step 4c symlink (you'll get 20 checkpoints over the full run, so disconnects only cost you the time since the last 5-epoch save). If Colab disconnects, run Steps 1, 2, 2b, 4c, then **Step 5b** to resume from the latest checkpoint.

**Before running, confirm:**
- Step 4 printed `bs_grid_spacing: 4.0` and `num_epochs: 100`
- `Runtime → Change runtime type` shows **A100 GPU** + **High-RAM ON**
- Step 4c printed both symlinks (checkpoints + data cache → Drive)
- You have **~20 GB free Drive space** for the raw HDF5 + checkpoints

In [ ]:
# Run training with data generation
import os
PROJECT_DIR = '/content/diffusion-posterior-sampling'
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError("Project folder missing! Re-run Step 2 (mount Drive + extract zip) and Step 2b.")
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}\n")

!python train_aoa_amp_building.py --generate_data --gpu 0 2>&1 | tee training_log.txt

## Step 5a: Save Progress to Drive (run anytime during training)

Open a **second tab** in Colab on the same runtime. Paste this cell to back up checkpoints to Drive while training continues in the first tab. Alternatively, run this cell after training completes in Step 5.

In [ ]:
import shutil, glob, os

DRIVE_SAVE = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_SAVE, exist_ok=True)

# Save latest checkpoint to Drive
ckpt_files = sorted(glob.glob('checkpoints/aoa_amp_building/checkpoint_epoch_*.pt'))
if ckpt_files:
    latest = ckpt_files[-1]
    dest = os.path.join(DRIVE_SAVE, os.path.basename(latest))
    shutil.copy2(latest, dest)
    print(f"Saved: {os.path.basename(latest)} ({os.path.getsize(latest)/1024**2:.0f} MB)")
else:
    print("No checkpoints found yet.")

# Save data cache (only first time, ~150 MB)
local_cache = 'data/building_training_randomized/all_building_configs_data_42.h5'
drive_cache = os.path.join(DRIVE_SAVE, 'all_building_configs_data_42.h5')
if os.path.exists(local_cache) and not os.path.exists(drive_cache):
    shutil.copy2(local_cache, drive_cache)
    print(f"Saved data cache ({os.path.getsize(local_cache)/1024**3:.2f} GB)")

# Save training log
if os.path.exists('training_log.txt'):
    shutil.copy2('training_log.txt', os.path.join(DRIVE_SAVE, 'training_log.txt'))
    print("Saved training_log.txt")

## Step 5b: Resume Training (if disconnected)

If Colab disconnects or you restart the session, run these cells **in order** before running the cell below:

1. **Step 1** — Check GPU
2. **Step 2** — Mount Drive, extract zip, cd, pip install
3. **Step 2b** — motionblur stub
4. **Step 4c** — Re-create the Drive symlink (so checkpoints keep auto-saving)

Then run the cell below — it resumes from the latest checkpoint on Drive and continues training.
**Do NOT run Step 3** (it deletes checkpoints) and **do NOT run Step 5** (it restarts from epoch 0).

In [ ]:
import shutil, glob, os

# Ensure we are in the project directory (kernel resets cwd after a restart)
PROJECT_DIR = '/content/diffusion-posterior-sampling'
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError("Project folder missing! Re-run Step 2 (mount Drive + extract zip) and Step 2b.")
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}\n")

DRIVE_SAVE = '/content/drive/MyDrive/thesis_results'

# --- Restore data cache from Drive (if local copy was lost) ---
local_cache = 'data/building_training_randomized/all_building_configs_data_42.h5'
drive_cache = os.path.join(DRIVE_SAVE, 'all_building_configs_data_42.h5')

if not os.path.exists(local_cache) and os.path.exists(drive_cache):
    os.makedirs('data/building_training_randomized', exist_ok=True)
    print(f"Restoring data cache from Drive ({os.path.getsize(drive_cache)/1024**3:.2f} GB)...")
    shutil.copy2(drive_cache, local_cache)
    print("Data cache restored.")
elif os.path.exists(local_cache):
    print(f"Data cache exists locally ({os.path.getsize(local_cache)/1024**3:.2f} GB).")
else:
    print("No data cache found. Will regenerate (add --generate_data).")

# --- Restore latest checkpoint from Drive ---
drive_ckpts = sorted(glob.glob(os.path.join(DRIVE_SAVE, 'checkpoints', 'checkpoint_epoch_*.pt')))
local_ckpts = sorted(glob.glob('checkpoints/aoa_amp_building/checkpoint_epoch_*.pt'))

if drive_ckpts and not local_ckpts:
    os.makedirs('checkpoints/aoa_amp_building', exist_ok=True)
    latest_drive = drive_ckpts[-1]
    dest = os.path.join('checkpoints/aoa_amp_building', os.path.basename(latest_drive))
    print(f"Restoring checkpoint: {os.path.basename(latest_drive)}")
    shutil.copy2(latest_drive, dest)
elif local_ckpts:
    print(f"Local checkpoint: {os.path.basename(local_ckpts[-1])}")
else:
    print("No checkpoint found - training will start from epoch 0.")

# --- Resume training ---
need_generate = "--generate_data" if not os.path.exists(local_cache) else ""
cmd = f"python train_aoa_amp_building.py --resume latest {need_generate} --gpu 0 2>&1 | tee -a training_log.txt"
print(f"\nRunning: {cmd}\n")
!{cmd}

## Step 6: Run DPS Inference

After training completes, reconstruct radio maps from 10% measurements.

In [ ]:
# Run inference on 16 test samples
import os
PROJECT_DIR = '/content/diffusion-posterior-sampling'
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError("Project folder missing! Re-run Step 2 (mount Drive + extract zip) and Step 2b.")
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}\n")

!python sample_condition_building.py --save_dir ./results --num_samples 16 --gpu 0

## Step 7: View Results

In [ ]:
import json
import numpy as np

# Load NMSE summary
metrics_files = glob.glob('results/inpainting/metrics/nmse_summary*.json')
if metrics_files:
    with open(metrics_files[0]) as f:
        summary = json.load(f)
    
    ch_labels = ['AoA1', 'AoA2', 'AoA3', 'Amp1', 'Amp2', 'Amp3',
                 'sin1', 'sin2', 'sin3', 'cos1', 'cos2', 'cos3']
    
    print(f"=== NMSE Results ({summary['samples']} samples, mask_prob={summary['mask_prob']}) ===")
    print(f"Total NMSE: {summary['avg_total_nmse']:.4f}")
    print(f"\nPer-channel NMSE:")
    for label, nmse in zip(ch_labels, summary['avg_channel_nmse']):
        bar = '#' * int(min(nmse * 50, 50))
        print(f"  {label:5s}: {nmse:.4f}  {bar}")
else:
    print("No metrics found. Run inference first (Step 6).")

In [ ]:
from IPython.display import display, Image as IPImage
from PIL import Image
import io

# Display comparison plots
pdf_files = sorted(glob.glob('results/inpainting/comparison/*.pdf'))
print(f"Found {len(pdf_files)} comparison plots\n")

# Show first 4 as PNG (convert from PDF)
try:
    import fitz  # PyMuPDF
    for pdf_path in pdf_files[:4]:
        doc = fitz.open(pdf_path)
        page = doc[0]
        pix = page.get_pixmap(dpi=150)
        img_data = pix.tobytes('png')
        print(f"--- {os.path.basename(pdf_path)} ---")
        display(IPImage(data=img_data))
        doc.close()
except ImportError:
    print("Install PyMuPDF to view PDFs inline: !pip install PyMuPDF")
    print("Or download the PDFs from results/inpainting/comparison/")

## Step 8: Save Results to Google Drive

Copy checkpoints and results back to Drive so you don't lose them.

In [ ]:
import shutil, os, glob

DRIVE_SAVE = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_SAVE, exist_ok=True)


def _safe_copy(src, dst, label):
    """Copy src -> dst, but skip if they resolve to the same file (e.g. when
    the source is a symlink that already points at dst, as set up in Step 4c)."""
    if not os.path.exists(src):
        return
    src_real = os.path.realpath(src)
    dst_real = os.path.realpath(dst) if os.path.exists(dst) else None
    if src_real == dst_real:
        print(f"{label}: already on Drive at {dst} (symlinked) - skipping copy")
        return
    shutil.copy2(src, dst)
    size_gb = os.path.getsize(src) / 1024**3
    print(f"{label}: saved {dst} ({size_gb:.2f} GB)")


# Copy final checkpoint (and intermediates) - checkpoints/ is already a symlink
# to Drive via Step 4c, so these are typically no-ops, but we still report.
ckpt_files = sorted(glob.glob('checkpoints/aoa_amp_building/checkpoint_epoch_*.pt'))
if ckpt_files:
    latest = ckpt_files[-1]
    dest = os.path.join(DRIVE_SAVE, 'checkpoints', os.path.basename(latest))
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    _safe_copy(latest, dest, label=f"Final checkpoint ({os.path.basename(latest)})")

# Copy results folder
if os.path.exists('results'):
    dest_results = os.path.join(DRIVE_SAVE, 'results')
    if os.path.exists(dest_results):
        shutil.rmtree(dest_results)
    shutil.copytree('results', dest_results)
    print(f"Results: saved to {dest_results}")

# Copy training log
if os.path.exists('training_log.txt'):
    shutil.copy2('training_log.txt', os.path.join(DRIVE_SAVE, 'training_log.txt'))
    print("Training log: saved")

# Copy data cache (skipped if already symlinked to Drive)
data_cache = 'data/building_training_randomized/all_building_configs_data_42.h5'
dest_data = os.path.join(DRIVE_SAVE, 'all_building_configs_data_42.h5')
_safe_copy(data_cache, dest_data, label="Data cache")

print(f"\nAll saved to: {DRIVE_SAVE}")